In [1]:
import sys
sys.path.append('..')
from scripts.get_data import get_data
from scripts.params import *
from google.cloud import bigquery

/Users/titouan/.pyenv/versions/3.10.6/envs/monitor-reactor/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.6) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
print("Dataset =", BQ_DATASET)
print("Project =", GCP_PROJECT_NAME)


Dataset = TEP_Faulty_Training
Project = monitor-the-reactor


In [3]:
faulty_test = get_data(
    col_to_keep=COLUMN_NAMES)

KeyboardInterrupt: 

In [ ]:
def get_data_test(
    project_id=GCP_PROJECT_NAME,
    dataset=BQ_FAULTY_TRAIN,
    col_to_keep=0,
    col_to_drop=0,
    sample_division=SAMPLE_DIVISION,
    fault=1
) :
    fault_intervall = (
        f"{(fault-1)*500/sample_division} AND "
        f"{(fault-1)*500/sample_division + 500/sample_division}"
)

    if col_to_keep == 0:

        col_left = tuple(set(COLUMN_NAMES) - set(col_to_drop))

        query = f"""
        SELECT {', '.join(col_left)}
        FROM `{project_id}`.`{dataset}`.`csv`
        WHERE sample BETWEEN {fault_intervall}
        AND MOD(sample, {sample_division}) = 0
        ORDER BY faultNumber, simulationRun, sample
        """

        client = bigquery.Client(project=project_id, location="EU")
        query_job = client.query(query)
        result = query_job.result()
        df = result.to_dataframe()

        return df

    elif col_to_drop == 0:
        query = f"""
        SELECT {', '.join(col_to_keep)}
        FROM `{project_id}`.`{dataset}`.`csv`
        WHERE sample BETWEEN {fault_intervall}
        AND MOD(sample, {sample_division}) = 0
        ORDER BY faultNumber, simulationRun, sample
        """

        client = bigquery.Client(project=project_id, location="EU")
        query_job = client.query(query)
        result = query_job.result()
        df = result.to_dataframe()

        return df

In [27]:
get_data_test(col_to_keep=COLUMN_NAMES)

BadRequest: 400 Syntax error: Unexpected keyword AND at [5:9]; reason: invalidQuery, location: query, message: Syntax error: Unexpected keyword AND at [5:9]

Location: EU
Job ID: d4ed7542-9540-46a0-be2b-4573816265dd
